# E02: 
split up the dataset randomly into 80% train set, 10% dev set, 10% test set. Train the bigram and trigram models only on the training set. Evaluate them on dev and test splits. What can you see?

# Solution:

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [2]:
words = open("names.txt", 'r').read().splitlines()

In [3]:
words[:5]

['emma', 'olivia', 'ava', 'isabella', 'sophia']

In [4]:
len(words)

32033

In [5]:
min(words, key=len), len(min(words, key=len))

('an', 2)

In [6]:
max(words, key=len), len(max(words, key=len))

('muhammadibrahim', 15)

In [7]:
# Get list of all the characters
chars = ['.'] + sorted(list(set(''.join(words)))) # ., a, b, c, d, .... x, y, z
len(chars)

27

In [8]:
# Create dictionary for mapping single character to its ID
stoi = {s:i for i, s in enumerate(chars)}

# Create another dictionary for reverse mapping
itos = {i:s for s, i in stoi.items()}

# Check if its correct
stoi['.'], stoi['a'], itos[0]

(0, 1, '.')

**Note:** A universal dictionary (`combinations + chars`) would not work for Trigram because the character IDs would fall in the range **729–755**, while the output classes are expected to be in the range **0–26**.

Doing it the other way around (`chars + combinations`) would not work either, because the combination IDs would fall in the range **27–755**, while `one_hot` for the input expects IDs in the range **0–728** when `num_classes=729`.

Therefore, separate dictionaries are needed:
- `ctoi` → character → IDs `0–26`
- `stoi` → combination → IDs `0–728`


In [9]:
# Get all 27*27 combinations (26 alphabets and one '.')
all_combinations = [a + b for a in chars for b in chars]

# Create dictionary for mapping combination to its ID
ctoi={c:i for i,c in enumerate(all_combinations)}

# Create another dictionary for reverse mapping
itoc={i:c for c, i in ctoi.items()}

# Check if its correct
len(ctoi), ctoi['..'], itoc[1]

(729, 0, '.a')

## Create train, dev and test split

In [10]:
g1 = torch.Generator().manual_seed(42)

perm = torch.randperm(len(words), generator=g1)
perm, perm.shape

(tensor([ 4348, 12372,  7029,  ...,  7956,  2399,  8375]), torch.Size([32033]))

In [11]:
words_shuffled = [words[i] for i in perm]

train_words = words_shuffled[:25626]
dev_words   = words_shuffled[25626:28830]
test_words  = words_shuffled[28830:32033]

## Bigram

In [12]:
# Creating the dataset
def create_dataset_bigram(words):
    xs, ys = [], []
    for w in words:
        chs = ['.'] + list(w) + ['.']  # eg ['.', 'e', 'm', 'm', 'a', '.']
        for ch1, ch2 in zip(chs, chs[1:]):
            idx1 = stoi[ch1]
            idx2 = stoi[ch2]
            xs.append(idx1)
            ys.append(idx2)
    
    # Converting dataset into tensor
    xs = torch.tensor(xs)
    ys = torch.tensor(ys)
    num = xs.numel()
    return xs, ys, num 

In [13]:
xs_train_b, ys_train_b, training_size_b = create_dataset_bigram(train_words)
xs_dev_b, ys_dev_b, dev_size_b = create_dataset_bigram(dev_words)
xs_test_b, ys_test_b, test_size_b = create_dataset_bigram(test_words)

In [14]:
# Testing the created dataset for correctness
def unit_test_dataset(xs, ys):
    for i in range(10):
        if itos[ys[i].item()]=='.':
            print(itos[xs[i].item()], itos[ys[i].item()], sep="")
            break
        print(itos[xs[i].item()], itos[ys[i].item()], sep="")


## Trainset test
print("-------------First sample in trainset--------------")
unit_test_dataset(xs_train_b, ys_train_b)
print(f"Trainset size: {training_size_b}")
print("---------------------------------------------------")

## Devset test
print("-------------First sample in devset----------------")
unit_test_dataset(xs_dev_b, ys_dev_b)
print(f"Devset size: {dev_size_b}")
print("---------------------------------------------------")

## Testset test
print("-------------First sample in testset----------------")
unit_test_dataset(xs_test_b, ys_test_b)
print(f"Testset size: {test_size_b}")
print("---------------------------------------------------")

-------------First sample in trainset--------------
.e
ed
di
is
so
on
n.
Trainset size: 182819
---------------------------------------------------
-------------First sample in devset----------------
.x
xa
av
vi
ie
er
ra
a.
Devset size: 22768
---------------------------------------------------
-------------First sample in testset----------------
.j
jo
os
se
et
tt
te
e.
Testset size: 22559
---------------------------------------------------


In [15]:
# Setting up the generator and initializing weights with random values
g_bigram = torch.Generator().manual_seed(312313)
W_bigram = torch.randn((27, 27), generator=g_bigram, requires_grad=True)

In [16]:
# Training loop
for i in range(50):
    #----------Forward pass------------
    xenc = F.one_hot(xs_train_b, num_classes=27).float() #27
    logits = xenc @ W_bigram # (182819, 27) @ (27, 27) = (182819,27)

    counts = logits.exp() 
    probs = counts/counts.sum(1, keepdim=True) 
    loss = -probs[torch.arange(training_size_b), ys_train_b].log().mean()
    print(loss.item())
    
    #-----------Backward pass--------------
    W_bigram.grad = None # set weights to zero
    loss.backward()

    #------------Update weights--------------
    W_bigram.data += -70 * W_bigram.grad

3.867845058441162
3.297778367996216
3.009648323059082
2.87753963470459
2.7926032543182373
2.7344672679901123
2.693356513977051
2.6631019115448
2.639805793762207
2.6211113929748535
2.6056478023529053
2.592590093612671
2.5814034938812256
2.5717146396636963
2.5632483959198
2.555795431137085
2.5491933822631836
2.5433130264282227
2.5380496978759766
2.5333173274993896
2.5290448665618896
2.525172710418701
2.521650791168213
2.518435001373291
2.5154898166656494
2.5127835273742676
2.5102882385253906
2.507982015609741
2.5058436393737793
2.5038561820983887
2.5020041465759277
2.500274419784546
2.498655319213867
2.4971365928649902
2.4957096576690674
2.494366407394409
2.4930999279022217
2.491903781890869
2.490772247314453
2.4897007942199707
2.488684892654419
2.487719774246216
2.486802577972412
2.485929489135742
2.4850971698760986
2.4843039512634277
2.483546257019043
2.4828224182128906
2.4821300506591797
2.4814672470092773


In [17]:
# Sampling names from the 'neural net' to get names
g2 = torch.Generator().manual_seed(2147483647)

for i in range(5):
  
  out = []
  ix = 0
  while True:
    
    # ----------
    # BEFORE:
    #p = P[ix]
    # ----------
    # NOW:
    xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
    logits = xenc @ W_bigram # predict log-counts
    counts = logits.exp() # counts, equivalent to N
    p = counts / counts.sum(1, keepdims=True) # probabilities for next character
    # ----------
    
    ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g2).item()
    out.append(itos[ix])
    if ix == 0:
      break
  print(''.join(out))

junide.
janasah.
prelay.
a.
nn.


## Evaluating Bigram

In [31]:
# Evaluating on dev_set 
xenc = F.one_hot(xs_dev_b, num_classes=27).float()
logits = xenc @ W_bigram
counts = logits.exp()
probs = counts / counts.sum(1, keepdim=True)
logprobs = torch.log(probs)
nlls = -logprobs[torch.arange(len(ys_dev_b)), ys_dev_b]

avg_nll_bigram_dev = nlls.mean().item()

print('=========')
print(f'average negative log likelihood, i.e. loss = {avg_nll_bigram_dev}')



# Evaluating on test_set 
xenc = F.one_hot(xs_test_b, num_classes=27).float()
logits = xenc @ W_bigram
counts = logits.exp()
probs = counts / counts.sum(1, keepdim=True)
logprobs = torch.log(probs)
nlls = -logprobs[torch.arange(len(ys_test_b)), ys_test_b]

avg_nll_bigram_test = nlls.mean().item()

print('=========')
print(f'average negative log likelihood, i.e. loss = {avg_nll_bigram_test}')



average negative log likelihood, i.e. loss = 2.487985849380493
average negative log likelihood, i.e. loss = 2.4753546714782715


## Trigram

In [20]:
# Creating the dataset
def create_dataset_trigram(words):
    xs, ys = [], []
    for w in words:
        chs = ['.','.'] + list(w) + ['.']  # eg ['.', '.', 'e', 'm', 'm', 'a', '.']
        for ch1, ch2, ch3 in zip(chs, chs[1:], chs[2:]):
            ch12 = ch1+ch2
            idx12 = ctoi[ch12]
            idx3 = stoi[ch3]
            xs.append(idx12)
            ys.append(idx3)
    
    # Converting dataset into tensor
    xs = torch.tensor(xs)
    ys = torch.tensor(ys)
    num = xs.numel()
    return xs, ys, num 

In [21]:
xs_train_t, ys_train_t, training_size_t = create_dataset_trigram(train_words)
xs_dev_t, ys_dev_t, dev_size_t = create_dataset_trigram(dev_words)
xs_test_t, ys_test_t, test_size_t = create_dataset_trigram(test_words)

In [22]:
print(xs_dev_t.shape)
print(ys_dev_t.shape)
print(dev_size_t)

torch.Size([22768])
torch.Size([22768])
22768


In [23]:
# Testing the created dataset for correctness
def unit_test_dataset(xs, ys):
    for i in range(10):
        if itos[ys[i].item()]=='.':
            print(itoc[xs[i].item()], itos[ys[i].item()], sep="")
            break
        print(itoc[xs[i].item()], itos[ys[i].item()], sep="")

## Trainset test
print("-------------First sample in trainset--------------")
unit_test_dataset(xs_train_t, ys_train_t)
print(f"Trainset size: {training_size_t}")
print("---------------------------------------------------")

## Devset test
print("-------------First sample in devset----------------")
unit_test_dataset(xs_dev_t, ys_dev_t)
print(f"Devset size: {dev_size_t}")
print("---------------------------------------------------")

## Testset test
print("-------------First sample in testset----------------")
unit_test_dataset(xs_test_t, ys_test_t)
print(f"Testset size: {test_size_t}")
print("---------------------------------------------------")

-------------First sample in trainset--------------
..e
.ed
edi
dis
iso
son
on.
Trainset size: 182819
---------------------------------------------------
-------------First sample in devset----------------
..x
.xa
xav
avi
vie
ier
era
ra.
Devset size: 22768
---------------------------------------------------
-------------First sample in testset----------------
..j
.jo
jos
ose
set
ett
tte
te.
Testset size: 22559
---------------------------------------------------


In [24]:
# Setting up the generator and initializing weights with random values
g = torch.Generator().manual_seed(312312)
W_trigram = torch.randn((729, 27), generator=g, requires_grad=True)

In [25]:
# Training loop
for i in range(50):
    #----------Forward pass------------
    xenc = F.one_hot(xs_train_t, num_classes=729).float() #729
    logits = xenc @ W_trigram # (182819, 729) @ (729, 27) = (182819,27)

    counts = logits.exp() 
    probs = counts/counts.sum(1, keepdim=True) 
    loss = -probs[torch.arange(training_size_t), ys_train_t].log().mean()
    print(loss.item())
    
    #-----------Backward pass--------------
    W_trigram.grad = None # set weights to zero
    loss.backward()

    #------------Update weights--------------
    W_trigram.data += -70 * W_trigram.grad

3.788719892501831
3.610229253768921
3.501995801925659
3.420778512954712
3.352067708969116
3.291569232940674
3.2376551628112793
3.189673900604248
3.1470437049865723
3.109065532684326
3.07499361038208
3.0441579818725586
3.0160202980041504
2.9901700019836426
2.966292142868042
2.9441380500793457
2.923503875732422
2.9042181968688965
2.8861331939697266
2.869124412536621
2.8530831336975098
2.837916851043701
2.823544502258301
2.809896230697632
2.7969119548797607
2.7845373153686523
2.7727255821228027
2.7614359855651855
2.7506308555603027
2.740276336669922
2.7303435802459717
2.7208046913146973
2.7116358280181885
2.7028136253356934
2.6943180561065674
2.686129331588745
2.6782305240631104
2.670605182647705
2.663238286972046
2.6561155319213867
2.649224042892456
2.642552137374878
2.6360886096954346
2.6298232078552246
2.623745918273926
2.617847442626953
2.612119674682617
2.6065549850463867
2.6011457443237305
2.5958855152130127


**Note: Importance of `keepdim=True`**

`logits.sum(1, keepdim=True).shape` → `torch.Size([4, 1])`  

If we forget `keepdim=True`, PyTorch removes the summed dimension, giving a tensor of shape `[4]`.

During broadcasting with `logits` of shape `[4, 27]`, the `[4]` tensor is aligned with the **last dimension**, so it is treated as `[1, 4]`, not `[4, 1]`.

Thus, `[4, 27] / [1, 4]` is not compatible for broadcasting, whereas `[4, 27] / [4, 1]` broadcasts correctly.

Therefore, `keepdim=True` is necessary to preserve the dimension as `[4, 1]` and broadcast the sum across the 27 classes for each example.

In [27]:
# Generating names
for i in range(10):
    out = [] # for storing the output
    context = ['.', '.'] # for triggering the model to start outputting characters to form a name
    
    while True:
        pair = context[0] + context[1]
        ix = ctoi[pair]
    
        xenc = F.one_hot(torch.tensor([ix]), num_classes=729).float() 
        logits = xenc @ W_trigram
        counts = logits.exp()
        p = counts / counts.sum(1, keepdim=True)

        # Randomly draw one index from the 27 indices, using the values in p as the sampling probabilities.
        # idx with highest probability has higher chances, but its not guaranteed
        next_ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
    
        next_char = itos[next_ix]

        if next_char == '.': 
            break         # breaking when it hits end character
        out.append(next_char)
        context = [context[1], next_char]
    
    if len(out)>1:
        print(''.join(out)) 

doexdcrayvmhkwtbycwelioniah
indkihtvfltejdknygtffyxaghoyhzujsha
kxjeavenn
kenelauri
hhmkkhlliddpguwuopeyczqqyepio
darier
kamarowjbnhbsgenduudkuhwwlotbfwvdbari
sknscrwomilytihi
ki
iohjedrqbxbbwckwa


## Evaluating Trigram

In [32]:
# Evaluating on dev_set 
xenc = F.one_hot(xs_dev_t, num_classes=729).float()
logits = xenc @ W_trigram
counts = logits.exp()
probs = counts / counts.sum(1, keepdim=True)
logprobs = torch.log(probs)
nlls = -logprobs[torch.arange(len(ys_dev_t)), ys_dev_t]

avg_nll_trigram_dev = nlls.mean().item()

print('=========')
print(f'average negative log likelihood, i.e. loss = {avg_nll_trigram_dev}')



# Evaluating on test_set 
xenc = F.one_hot(xs_test_t, num_classes=729).float()
logits = xenc @ W_trigram
counts = logits.exp()
probs = counts / counts.sum(1, keepdim=True)
logprobs = torch.log(probs)
nlls = -logprobs[torch.arange(len(ys_test_t)), ys_test_t]

avg_nll_trigram_test = nlls.mean().item()

print('=========')
print(f'average negative log likelihood, i.e. loss = {avg_nll_trigram_test}')

average negative log likelihood, i.e. loss = 2.6113967895507812
average negative log likelihood, i.e. loss = 2.596189498901367


## Evaluation Results: Bigram vs Trigram

In [42]:
print("------------------Results on devset-------------------")
print(f'Bigram loss  = {avg_nll_bigram_dev}')
print(f'Trigram loss = {avg_nll_trigram_dev}')
if avg_nll_trigram_dev>avg_nll_bigram_dev:
    print(f"Bigram wins by margin of {avg_nll_trigram_dev-avg_nll_bigram_dev}")
else:
    print(f"Trigram wins by margin of {avg_nll_bigram_dev-avg_nll_trigram_dev}")
print("-------------------------------------------------------")
print("------------------Results on testset-------------------")
print(f'Bigram loss  = {avg_nll_bigram_test}')
print(f'Trigram loss = {avg_nll_trigram_test}')
if avg_nll_trigram_test>avg_nll_bigram_test:
    print(f"Bigram wins by margin of {avg_nll_trigram_test-avg_nll_bigram_test}")
else:
    print(f"Trigram wins by margin of {avg_nll_bigram_test-avg_nll_trigram_test}")
print("-------------------------------------------------------")

------------------Results on devset-------------------
Bigram loss  = 2.487985849380493
Trigram loss = 2.6113967895507812
Bigram wins by margin of 0.12341094017028809
-------------------------------------------------------
------------------Results on testset-------------------
Bigram loss  = 2.4753546714782715
Trigram loss = 2.596189498901367
Bigram wins by margin of 0.1208348274230957
-------------------------------------------------------


With the same 50-iteration training setup and learning rate, the bigram model achieves lower NLL than the trigram model on both the training and held-out splits. The performance difference is consistent across train, dev, and test, so there is no obvious overfitting pattern. In this experiment, adding the second character of context did not improve performance.

**However, this does not establish that bigram is inherently better, because the larger trigram model may require different learning rate and more training to reach its full potential**